In [0]:
# Databricks notebook source
import os
import sys
import yaml
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.sgs_client import SGSClient, logger

# COMMAND ----------

# 1. Parâmetro do Job (Widget): aceita "diaria", "mensal" ou "todas"
dbutils.widgets.text("periodicidade_alvo", "todas", "Periodicidade (diaria/mensal/todas)")
periodicidade_alvo = dbutils.widgets.get("periodicidade_alvo").strip().lower()

caminho_config = os.path.join(repo_root, "config", "series_sgs.yaml")
with open(caminho_config, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

catalogo = config["catalogo"]
data_inicio_padrao = config["data_inicio_historico"]
data_hoje = datetime.today().strftime("%d/%m/%Y")
tabela_bronze = f"{catalogo}.bronze.sgs_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.gold")

# COMMAND ----------

# 2. Descoberta do Watermark (última data carregada por série na Bronze)
ultimas_datas = {}
if spark.catalog.tableExists(tabela_bronze):
    df_watermark = (
        spark.read.table(tabela_bronze)
        .withColumn("dt_ref", F.to_date(F.col("data"), "dd/MM/yyyy"))
        .groupBy("codigo_serie")
        .agg(F.max("dt_ref").alias("ultima_data"))
        .collect()
    )
    ultimas_datas = {row["codigo_serie"]: row["ultima_data"] for row in df_watermark if row["ultima_data"]}

# COMMAND ----------

# 3. Extração Incremental filtrada por periodicidade
client = SGSClient()
registros_bronze = []
timestamp_atual = datetime.now()

for serie in config["series"]:
    if not serie.get("ativo", False):
        continue

    per_serie = serie["periodicidade"].strip().lower()
    if periodicidade_alvo != "todas" and per_serie != periodicidade_alvo:
        continue

    codigo = int(serie["codigo"])
    ultima_dt = ultimas_datas.get(codigo)

    # Define a data inicial incremental com janela de segurança (lookback)
    if ultima_dt:
        dias_lookback = 7 if per_serie == "diaria" else 60
        dt_ini_calc = ultima_dt - relativedelta(days=dias_lookback)
        data_ini_extracao = dt_ini_calc.strftime("%d/%m/%Y")
    else:
        data_ini_extracao = data_inicio_padrao

    try:
        logger.info(f"Série {codigo} ({per_serie}): extração incremental de {data_ini_extracao} até {data_hoje}")
        dados_api = client.extrair_serie(
            codigo=codigo,
            data_inicial=data_ini_extracao,
            data_final=data_hoje
        )
        for item in dados_api:
            registros_bronze.append({
                "codigo_serie": codigo,
                "nome_serie": serie["nome"],
                "periodicidade": per_serie,
                "unidade": serie["unidade"],
                "data": str(item.get("data")),
                "valor": str(item.get("valor")),
                "timestamp_ingestao": timestamp_atual
            })
    except Exception as e:
        logger.error(f"Falha ao extrair a série {codigo} ({serie['nome']}): {e}")

# COMMAND ----------

# 4. Gravação Incremental (Append) na Bronze
schema_bronze = StructType([
    StructField("codigo_serie", IntegerType(), False),
    StructField("nome_serie", StringType(), True),
    StructField("periodicidade", StringType(), True),
    StructField("unidade", StringType(), True),
    StructField("data", StringType(), True),
    StructField("valor", StringType(), True),
    StructField("timestamp_ingestao", TimestampType(), False)
])

if registros_bronze:
    df_bronze = (
        spark.createDataFrame(registros_bronze, schema=schema_bronze)
        .withColumn("data_particao", F.to_date("timestamp_ingestao"))
    )
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .partitionBy("data_particao")
        .saveAsTable(tabela_bronze)
    )
    logger.info(f"Sucesso! {len(registros_bronze)} registros incrementais salvos em {tabela_bronze}.")
else:
    logger.info("Nenhum registro novo para processar.")

In [0]:
%sql
select * from economia.bronze.sgs_raw